# Frequency-dependent gain reduction — **LA2A** (SignalTrain)

A focused companion to `eval_la2a_nonlinearity.ipynb`. That notebook proves the
LA-2A is **non-LTI / time-varying** (best-LTI residual, gain-matched residual,
multiple coherence, bicoherence) — but *none* of its measures actually draw a
**frequency-dependent gain-reduction curve**. This notebook does exactly that.

### Why the LA-2A is not an ideal digital compressor
A textbook digital compressor applies a single **frequency-flat** gain `g(t)`
driven by a broadband detector: every frequency is attenuated by the same number
of dB, so its effective per-frequency gain `G(f)` is a flat line that does not
move with the operating point. The LA-2A is **optical** and breaks this on two
counts: (i) its T4 side-chain is **frequency-weighted** (not equally sensitive at
all frequencies), so the *amount* of gain reduction depends on where the energy
sits; and (ii) the T4/opto + tube stages add **frequency-dependent harmonic
coloration**, putting energy where the input had little. Both make the measured
`G(f) = Syy(f)/Sxx(f)` **non-flat and depth-dependent**. This notebook measures
the *shape and depth-dependence* of `G(f)` directly — letting the data, not an
assumed direction, show the tilt — because a black-box model fed only a broadband
GR envelope cannot reproduce any of it.

### Why a single broadband `g(t)` hides it
Within one STFT window the wet is `Y(f) ≈ g(f)·X(f)`. The other notebook's §2
removes a **single scalar** `g(t)` (broadband RMS) and lumps everything left —
the spectral tilt **and** the harmonic distortion — into one residual. To *see*
the frequency-dependence on its own we must measure the gain **per frequency**,
and to keep compression separate from no-compression we must measure it
**conditioned on the operating point**.

### What this notebook measures
The robust per-frequency gain is the power-weighted ratio
`G(f) = 10·log10( ΣSyy(f) / ΣSxx(f) )` (dB) — literally the input→output
**amplitude difference per frequency**, but power-weighted so silent bins don't
dominate. We compute it four ways:

1. **§A Level-conditioned gain family** — bin every STFT frame by its broadband
   GR depth `g_t`, accumulate `Sxx_k(f), Syy_k(f)` per depth bin `k`, and plot the
   family `G_k(f)`. The relative tilt `ΔG_k(f) = G_k(f) − ⟨G_k⟩_band` is the pure
   frequency-dependence: a horizontal line at 0 dB ⇒ frequency-flat compressor; a
   sloped/curved shape that **changes with depth** ⇒ the LA-2A signature.
2. **§B Tilt across all 42 settings** — `ΔG(f)` at each setting's deepest
   operating point, split Compress/Limit, coloured by peak-reduction.
3. **§C Per-band static I/O curves** — output-vs-input level for low/mid/high
   bands; a different knee/slope per band is frequency-dependent compression in
   the engineer-readable form.
4. **§D GR(f,t) spectrogram** — one excerpt, showing the gain reduction is
   non-uniform across frequency in time.

Dataset discovery and streaming reuse the **exact machinery** of the LA2A
coherence / nonlinearity notebooks.

In [ ]:
import os
import sys
import re
import glob
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

warnings.filterwarnings("ignore", category=UserWarning)

REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(REPO_ROOT / "06_output"))
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))

from eval_helpers import _read_dry_wet_segment, _pair_num_frames
from src.dsp_torch import gain_reduction_db, RMS_WINDOW

print(f"repo    : {REPO_ROOT}")
print(f"torch   : {torch.__version__}")
print(f"librosa : {librosa.__version__}")
print(f"RMS win : {RMS_WINDOW}")

In [ ]:
# SignalTrain LA-2A  (same discovery as eval_la2a_*.ipynb)
DATA_ROOT = "/Volumes/Saola's Drive/AllCode/thesis/data/LA2A"
SAMPLE_RATE = 44100

ANALYZE_SEC_PER_PAIR = 120.0
STREAM_CHUNK_SEC = 20.0

# --- STFT parameters ---
N_FFT = 4096
HOP_LENGTH = N_FFT // 4
WINDOW = "hann"
EPS = 1e-12

# Set to a small integer for smoke tests (limits pairs analysed)
MAX_EVAL_PAIRS = None

ALL_DIR = os.path.join(DATA_ROOT, "all")
assert os.path.isdir(ALL_DIR), f"Missing {ALL_DIR}"

_TARGET_RE = re.compile(r"target_(\d+)_LA2A_\dc__(\d+)__(\d+)\.wav$")


def discover_la2a_pairs(all_dir: str) -> list[dict]:
    pairs = []
    for tgt in sorted(glob.glob(os.path.join(all_dir, "target_*.wav"))):
        m = _TARGET_RE.search(os.path.basename(tgt))
        if not m:
            continue
        pid, comp, pr = m.group(1), int(m.group(2)), int(m.group(3))
        dry = os.path.join(all_dir, f"input_{pid}_.wav")
        if not os.path.isfile(dry):
            continue
        pairs.append({"id": pid, "comp": comp, "pr": pr, "setting": f"cl{comp}_pr{pr}",
                      "dry": dry, "wet": tgt})
    return sorted(pairs, key=lambda p: (p["comp"], p["pr"], p["id"]))


PAIRS = discover_la2a_pairs(ALL_DIR)
if MAX_EVAL_PAIRS is not None:
    PAIRS = PAIRS[:MAX_EVAL_PAIRS]

SETTING_PAIRS: dict[str, list[dict]] = {}
for p in PAIRS:
    SETTING_PAIRS.setdefault(p["setting"], []).append(p)
SETTINGS = sorted(SETTING_PAIRS, key=lambda s: (int(s.split("_pr")[0][2:]), int(s.split("_pr")[1])))


def setting_comp_pr(setting: str) -> tuple[int, int]:
    return int(setting.split("_pr")[0][2:]), int(setting.split("_pr")[1])


print(f"Pairs discovered : {len(PAIRS)}")
print(f"Settings         : {len(SETTINGS)}  "
      f"(Compress={sum(setting_comp_pr(s)[0]==0 for s in SETTINGS)}, "
      f"Limit={sum(setting_comp_pr(s)[0]==1 for s in SETTINGS)})")

chunk_frames = int(round(STREAM_CHUNK_SEC * SAMPLE_RATE))
analyze_frames = int(round(ANALYZE_SEC_PER_PAIR * SAMPLE_RATE))
FREQS = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=N_FFT)
print(f"Analyse {ANALYZE_SEC_PER_PAIR:.0f}s/pair | n_fft={N_FFT}, hop={HOP_LENGTH} "
      f"-> {len(FREQS)} freq bins")

## Configuration — bands, depth bins, styling

* **Broadband band** (operating-point detector): 200 Hz–8 kHz. Each STFT frame's
  broadband gain `g_t = 10·log10(ΣSyy_band / ΣSxx_band)` decides which **depth
  bin** it falls in.
* **Tilt band**: 100 Hz–10 kHz — the band over which `⟨G_k⟩` is taken when
  forming the relative tilt `ΔG_k(f)`.
* **I/O bands** (§C static curves): Low 60–250 Hz, Mid 500–2000 Hz, High
  4–10 kHz.
* **Summary scalar**: `HF−LF gain` = mean `G` over 4–8 kHz minus mean over
  100–300 Hz at a fixed deep operating point. More negative ⇒ stronger
  high-frequency-biased compression.

42 settings are coloured by **peak-reduction** (line style = mode), as in the
companion notebooks.

In [ ]:
# Frequency masks
F_MIN_PLOT, F_MAX_PLOT = 30.0, 16000.0
fmask     = (FREQS >= F_MIN_PLOT) & (FREQS <= F_MAX_PLOT)
BB_BAND   = (FREQS >= 200) & (FREQS <= 8000)      # broadband detector (operating point)
TILT_BAND = (FREQS >= 100) & (FREQS <= 10000)     # band-mean for relative tilt
HF_BAND   = (FREQS >= 4000) & (FREQS <= 8000)     # summary HF
LF_BAND   = (FREQS >= 100) & (FREQS <= 300)       # summary LF

IO_BANDS = [("Low 60-250 Hz",   (FREQS >= 60)   & (FREQS <= 250),   "#1f77b4"),
            ("Mid 0.5-2 kHz",   (FREQS >= 500)  & (FREQS <= 2000),  "#2ca02c"),
            ("High 4-10 kHz",   (FREQS >= 4000) & (FREQS <= 10000), "#d62728")]

# Operating-point (broadband GR depth) bins, dB. The hardest LA-2A limit setting
# averages ~-14 dB broadband GR, so the bins must resolve the full range (a too-
# shallow set crushes all the deep frames into one bin). Bins with too few frames
# are skipped when plotting.
DEPTH_EDGES = np.array([-30.0, -16.0, -13.0, -11.0, -9.0, -7.5, -6.0,
                        -4.5, -3.0, -1.5, 0.0, 5.0])
N_DEPTH = len(DEPTH_EDGES) - 1
MIN_FRAMES_PER_BIN = 200          # don't trust a depth curve below this

# 42 settings -> colour by peak-reduction, style by mode.
PR_NORM = Normalize(vmin=0, vmax=100)
PR_CMAP = plt.get_cmap("viridis")
DEPTH_CMAP = plt.get_cmap("plasma")
_MODE_HANDLES = [Line2D([0], [0], color="k", lw=1.4, ls="-",  label="Compress (cl0)"),
                 Line2D([0], [0], color="k", lw=1.4, ls="--", label="Limit (cl1)")]


def setting_style(setting):
    comp, pr = setting_comp_pr(setting)
    return PR_CMAP(PR_NORM(pr)), ("-" if comp == 0 else "--")


def severity(setting):
    comp, pr = setting_comp_pr(setting)
    return (pr, comp)


SETTINGS_BY_SEV = sorted(SETTINGS, key=severity, reverse=True)
HARDEST = SETTINGS_BY_SEV[0]
print(f"Hardest setting (by peak-reduction/mode): {HARDEST}")
print(f"Depth bins (dB): {DEPTH_EDGES}")

### Calibrate the I/O level axis

The §C static curves need fixed histogram edges. A short probe over the hardest
setting fixes a common dB range for the per-band input/output levels (the dry
material is shared across all settings, so the input axis is identical
everywhere).

In [ ]:
def _stft(sig):
    return librosa.stft(np.ascontiguousarray(sig, dtype=np.float32),
                        n_fft=N_FFT, hop_length=HOP_LENGTH, window=WINDOW, center=False)


# Probe one chunk of the hardest setting to fix the I/O level range.
_pp = SETTING_PAIRS[HARDEST][0]
_stop = min(_pair_num_frames(_pp["dry"], _pp["wet"], SAMPLE_RATE), chunk_frames)
_dry, _wet = _read_dry_wet_segment(_pp["dry"], _pp["wet"], 0, _stop, SAMPLE_RATE)
_L = min(_dry.shape[-1], _wet.shape[-1])
_PX = np.abs(_stft(_dry.squeeze(0).numpy()[:_L])) ** 2
_PY = np.abs(_stft(_wet.squeeze(0).numpy()[:_L])) ** 2
_levels = []
for _, m, _c in IO_BANDS:
    _levels.append(10 * np.log10(_PX[m].sum(0) + EPS))
    _levels.append(10 * np.log10(_PY[m].sum(0) + EPS))
_levels = np.concatenate(_levels)
_lo, _hi = np.percentile(_levels, 0.5) - 6, np.percentile(_levels, 99.5) + 6
IO_LEVEL_EDGES = np.arange(np.floor(_lo), np.ceil(_hi) + 1.0, 1.0)   # 1-dB bins
IO_LEVEL_CENTERS = 0.5 * (IO_LEVEL_EDGES[:-1] + IO_LEVEL_EDGES[1:])
print(f"I/O level axis: [{IO_LEVEL_EDGES[0]:.0f}, {IO_LEVEL_EDGES[-1]:.0f}] dB "
      f"in {len(IO_LEVEL_CENTERS)} bins")
del _PX, _PY, _dry, _wet, _levels; gc.collect()

## Level-conditioned spectral accumulator

One streaming pass per **setting**. For every STFT frame we:

1. compute the broadband gain `g_t = 10·log10(ΣSyy_band/ΣSxx_band)` and route the
   frame's `|X(f)|²`, `|Y(f)|²` into the matching **depth bin** `k`;
2. add to per-band 2-D **input×output level** histograms for the static curves.

From the accumulated `Sxx_k(f), Syy_k(f)` the per-depth gain curve is
`G_k(f) = 10·log10(Syy_k/Sxx_k)`. Power-weighting makes silent bins harmless; no
per-bin ratios are ever taken on near-zero energy.

In [ ]:
class FreqDepAccumulator:
    '''Level-conditioned per-frequency gain + per-band static I/O histograms.'''

    def __init__(self, n_bins):
        self.Sxx_k = np.zeros((N_DEPTH, n_bins), np.float64)
        self.Syy_k = np.zeros((N_DEPTH, n_bins), np.float64)
        self.nframes_k = np.zeros(N_DEPTH, np.int64)
        self.gsum_k = np.zeros(N_DEPTH, np.float64)        # for mean depth label
        self.Sxx = np.zeros(n_bins, np.float64)
        self.Syy = np.zeros(n_bins, np.float64)
        self.n_frames = 0
        nlev = len(IO_LEVEL_CENTERS)
        self.io_hist = [np.zeros((nlev, nlev), np.float64) for _ in IO_BANDS]

    def add(self, x, y):
        if min(len(x), len(y)) < N_FFT:
            return
        PX = np.abs(_stft(x)) ** 2          # [F, T]
        PY = np.abs(_stft(y)) ** 2
        bx = PX[BB_BAND].sum(0)             # [T] broadband input / output power
        by = PY[BB_BAND].sum(0)
        g_db = 10 * np.log10((by + EPS) / (bx + EPS))      # broadband per-frame gain
        idx = np.digitize(g_db, DEPTH_EDGES) - 1
        for k in range(N_DEPTH):
            sel = idx == k
            n = int(sel.sum())
            if n:
                self.Sxx_k[k] += PX[:, sel].sum(1)
                self.Syy_k[k] += PY[:, sel].sum(1)
                self.nframes_k[k] += n
                self.gsum_k[k] += g_db[sel].sum()
        self.Sxx += PX.sum(1)
        self.Syy += PY.sum(1)
        self.n_frames += PX.shape[1]
        for bi, (_, m, _c) in enumerate(IO_BANDS):
            in_db = 10 * np.log10(PX[m].sum(0) + EPS)
            out_db = 10 * np.log10(PY[m].sum(0) + EPS)
            H, _, _ = np.histogram2d(in_db, out_db, bins=[IO_LEVEL_EDGES, IO_LEVEL_EDGES])
            self.io_hist[bi] += H

    # ---- per-frequency gain curves ----
    def gain_curve(self, k):
        return 10 * np.log10((self.Syy_k[k] + EPS) / (self.Sxx_k[k] + EPS))

    def gain_overall(self):
        return 10 * np.log10((self.Syy + EPS) / (self.Sxx + EPS))

    def rel_tilt(self, k, band=TILT_BAND):
        g = self.gain_curve(k)
        return g - np.mean(g[band])

    def mean_depth(self, k):
        return self.gsum_k[k] / max(self.nframes_k[k], 1)

    def populated_bins(self, min_frames=MIN_FRAMES_PER_BIN):
        return [k for k in range(N_DEPTH) if self.nframes_k[k] >= min_frames]

    def bin_deepest(self, min_frames=MIN_FRAMES_PER_BIN):
        '''Deepest (most negative) well-populated operating bin — each setting's
        hardest-working point. Returns None if nothing clears min_frames.'''
        pops = self.populated_bins(min_frames)
        if not pops:
            return None
        return min(pops, key=self.mean_depth)

    # ---- static I/O transfer ----
    def io_curve(self, bi):
        H = self.io_hist[bi]                          # [in, out]
        w = H.sum(1)                                  # frames per input bin
        mean_out = (H * IO_LEVEL_CENTERS[None, :]).sum(1) / (w + EPS)
        return IO_LEVEL_CENTERS, mean_out, w

    # ---- summary scalar ----
    def hf_lf_gain(self, k):
        g = self.gain_curve(k)
        return float(np.mean(g[HF_BAND]) - np.mean(g[LF_BAND]))


@torch.no_grad()
def accumulate_setting(setting, acc, max_frames):
    for p in SETTING_PAIRS[setting]:
        total = min(_pair_num_frames(p["dry"], p["wet"], SAMPLE_RATE), max_frames)
        for o in range(0, total, chunk_frames):
            stop = min(o + chunk_frames, total)
            dry, wet = _read_dry_wet_segment(p["dry"], p["wet"], o, stop, SAMPLE_RATE)
            L = min(dry.shape[-1], wet.shape[-1])
            if L < N_FFT:
                continue
            acc.add(dry.squeeze(0).numpy()[:L], wet.squeeze(0).numpy()[:L])
        gc.collect()


results = {}
for si, setting in enumerate(SETTINGS, start=1):
    acc = FreqDepAccumulator(len(FREQS))
    spairs = SETTING_PAIRS[setting]
    print(f"[{si}/{len(SETTINGS)}] {setting}  ({len(spairs)} pair(s))")
    accumulate_setting(setting, acc, analyze_frames)
    results[setting] = {"acc": acc, "pairs": spairs}

print("\nDepth-bin frame counts (hardest setting):")
_a = results[HARDEST]["acc"]
for k in range(N_DEPTH):
    print(f"  bin {k} (~{_a.mean_depth(k):6.2f} dB): {_a.nframes_k[k]:>7d} frames")

## §A — Level-conditioned per-frequency gain (the frequency-dependent GR curve)

For the **hardest** setting, each curve is the per-frequency gain `G_k(f)` at one
broadband operating depth (colour = depth). **Left**: absolute gain — every curve
sits near its broadband depth, but the *shape* is not flat. **Right**: the
relative tilt `ΔG_k(f)` with the broadband level removed — this is the pure
frequency-dependence. A digital frequency-flat compressor would give horizontal
lines at 0 dB on the right; the LA-2A's curves are **not flat and their shape
shifts with operating depth** — the direction and size of the tilt are what the
plot reveals (note an upward HF tilt can reflect added HF harmonic energy as much
as reduced HF attenuation).

In [ ]:
acc = results[HARDEST]["acc"]
bins = acc.populated_bins()
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for k in bins:
    col = DEPTH_CMAP((acc.mean_depth(k) - DEPTH_EDGES[0]) / (DEPTH_EDGES[-1] - DEPTH_EDGES[0]))
    lab = f"{acc.mean_depth(k):.1f} dB"
    axes[0].semilogx(FREQS[fmask], acc.gain_curve(k)[fmask], color=col, lw=1.4, label=lab)
    axes[1].semilogx(FREQS[fmask], acc.rel_tilt(k)[fmask],  color=col, lw=1.4, label=lab)
axes[0].set_title(f"Absolute per-frequency gain  $G_k(f)$  —  {HARDEST}")
axes[0].set_ylabel(r"gain  $10\log_{10}(S_{yy}/S_{xx})$  (dB)")
axes[1].axhline(0, color="k", lw=0.8, ls=":")
axes[1].set_title(r"Relative tilt  $\Delta G_k(f)=G_k(f)-\langle G_k\rangle$  (frequency-dependence)")
axes[1].set_ylabel(r"$\Delta G_k(f)$  (dB)")
for ax in axes:
    ax.set_xlabel("frequency (Hz)")
    ax.set_xlim(F_MIN_PLOT, F_MAX_PLOT)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(title="operating depth", fontsize=8, ncol=2)
fig.suptitle("LA2A §A  Frequency-dependent gain reduction, conditioned on operating depth", y=1.02)
plt.tight_layout(); plt.show()

## §B — Spectral tilt across all 42 settings

`ΔG(f)` evaluated at **each setting's own deepest well-populated operating bin**
(its hardest-working point), split Compress / Limit and coloured by peak-reduction.
Hard and soft settings live at very different broadband depths — almost none share
a single absolute depth — so each is shown where it actually compresses. A
consistent tilt shape across settings is the device-level frequency-dependent GR
signature; its *magnitude* should grow with peak-reduction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
n_used = 0
for ax, comp in zip(axes, (0, 1)):
    for setting in SETTINGS:
        c, pr = setting_comp_pr(setting)
        if c != comp:
            continue
        a = results[setting]["acc"]
        k = a.bin_deepest()
        if k is None:
            continue
        ax.semilogx(FREQS[fmask], a.rel_tilt(k)[fmask], color=PR_CMAP(PR_NORM(pr)), lw=1.0, alpha=0.85)
        n_used += 1
    ax.axhline(0, color="k", lw=0.8, ls=":")
    ax.set_xlabel("frequency (Hz)")
    ax.set_title("Compress (cl0)" if comp == 0 else "Limit (cl1)")
    ax.set_xlim(F_MIN_PLOT, F_MAX_PLOT)
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel(r"relative tilt  $\Delta G(f)$  (dB)")
fig.suptitle(f"LA2A §B  Spectral tilt at each setting's deepest operating point "
             f"({n_used} settings)")
sm = ScalarMappable(norm=PR_NORM, cmap=PR_CMAP); sm.set_array([])
fig.colorbar(sm, ax=axes, label="Peak Reduction", pad=0.015)
plt.show()

## §C — Per-band static input→output transfer curves

Output level vs input level for the Low / Mid / High bands (hardest setting),
each the histogram-averaged curve over all analysed frames. The dotted line is
unity (no compression). **Bands that bend away from unity sooner / harder are
compressed more** — a different effective threshold and ratio per frequency,
which is frequency-dependent compression in classic compressor terms. The shaded
density shows where the material actually sits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
acc = results[HARDEST]["acc"]
lo, hi = IO_LEVEL_EDGES[0], IO_LEVEL_EDGES[-1]
# left: density of the Mid band for context; right would be busy with 3 -> overlay means.
for bi, (name, m, col) in enumerate(IO_BANDS):
    cin, mout, w = acc.io_curve(bi)
    good = w > (0.002 * w.sum())          # drop sparsely populated input bins
    axes[1].plot(cin[good], mout[good], color=col, lw=2.0, label=name)
axes[1].plot([lo, hi], [lo, hi], "k:", lw=1.0, label="unity (no comp.)")
axes[1].set_xlim(lo, hi); axes[1].set_ylim(lo, hi)
axes[1].set_aspect("equal")
axes[1].set_xlabel("input band level (dB)")
axes[1].set_ylabel("output band level (dB)")
axes[1].set_title("Static I/O transfer per band")
axes[1].grid(True, alpha=0.3); axes[1].legend(fontsize=8)

# left: the Mid-band 2-D density behind its mean curve
mid_bi = 1
H = acc.io_hist[mid_bi]
axes[0].pcolormesh(IO_LEVEL_EDGES, IO_LEVEL_EDGES, np.log10(H.T + 1.0), cmap="Greys", shading="auto")
cin, mout, w = acc.io_curve(mid_bi)
good = w > (0.002 * w.sum())
axes[0].plot(cin[good], mout[good], color=IO_BANDS[mid_bi][2], lw=2.0, label="mean")
axes[0].plot([lo, hi], [lo, hi], "k:", lw=1.0, label="unity")
axes[0].set_xlim(lo, hi); axes[0].set_ylim(lo, hi); axes[0].set_aspect("equal")
axes[0].set_xlabel("input band level (dB)"); axes[0].set_ylabel("output band level (dB)")
axes[0].set_title(f"{IO_BANDS[mid_bi][0]} density (log frames)")
axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=8)
fig.suptitle(f"LA2A §C  Frequency-dependent static compression characteristic — {HARDEST}", y=1.02)
plt.tight_layout(); plt.show()

## §D — Frequency-dependent GR spectrogram (one excerpt)

`GR(f,t) = 10·log10(|Y(f,t)|²/|X(f,t)|²)` for a short excerpt of the hardest
setting, masked to bins with enough input energy (silent bins → blank). The
broadband per-frame gain is overlaid as a white line. If the gain reduction were
frequency-flat the map would be vertically uniform (one colour per time column);
any **vertical structure** — bands diverging from the broadband line — is the
frequency-dependence in action.

In [ ]:
SPEC_OFFSET_SEC, SPEC_SECONDS = 30.0, 12.0
p = SETTING_PAIRS[HARDEST][0]
total = _pair_num_frames(p["dry"], p["wet"], SAMPLE_RATE)
o = min(int(SPEC_OFFSET_SEC * SAMPLE_RATE), max(total - int(SPEC_SECONDS * SAMPLE_RATE), 0))
stop = min(o + int(SPEC_SECONDS * SAMPLE_RATE), total)
dry, wet = _read_dry_wet_segment(p["dry"], p["wet"], o, stop, SAMPLE_RATE)
L = min(dry.shape[-1], wet.shape[-1])
PX = np.abs(_stft(dry.squeeze(0).numpy()[:L])) ** 2
PY = np.abs(_stft(wet.squeeze(0).numpy()[:L])) ** 2
gr = 10 * np.log10((PY + EPS) / (PX + EPS))
# energy mask: hide bins where the input is far below the frame's peak
thresh = PX.max(0, keepdims=True) * 1e-4
gr_masked = np.where(PX > thresh, gr, np.nan)
times = np.arange(PX.shape[1]) * HOP_LENGTH / SAMPLE_RATE
g_bb = 10 * np.log10((PY[BB_BAND].sum(0) + EPS) / (PX[BB_BAND].sum(0) + EPS))

fig, ax = plt.subplots(figsize=(14, 5.5))
pm = ax.pcolormesh(times, FREQS[fmask], gr_masked[fmask], cmap="RdBu",
                   vmin=-18, vmax=6, shading="auto")
ax.set_yscale("log"); ax.set_ylim(F_MIN_PLOT, F_MAX_PLOT)
ax.set_xlabel("time (s)"); ax.set_ylabel("frequency (Hz)")
ax.set_title(f"LA2A §D  Per-frequency gain reduction  GR(f,t) — {HARDEST} (pair {p['id']})")
fig.colorbar(pm, ax=ax, label="GR (dB)  [blue = attenuated]")
ax2 = ax.twinx()
ax2.plot(times, g_bb, color="white", lw=1.2, alpha=0.9)
ax2.set_ylabel("broadband gain (dB)", color="0.3")
ax2.set_ylim(-18, 6)
plt.tight_layout(); plt.show()
del PX, PY, gr, gr_masked; gc.collect()

## Summary

Per setting (hardest → softest): the frequency-dependent GR strength at each
setting's deepest operating point. `HF−LF gain` < 0 means the highs are attenuated
more than the lows (the LA-2A behaviour); a frequency-flat digital compressor
would sit at ~0 dB. `tilt slope` is the least-squares slope of `ΔG(f)` vs
`log10(f)` over the tilt band (dB per decade; negative = darker with compression).
Note `HF−LF gain` also folds in HF harmonic energy added by the box, so read it
together with §A/§B shapes rather than as a pure attenuation difference.

In [ ]:
logf = np.log10(FREQS[TILT_BAND] + EPS)
rows = []
for setting in SETTINGS_BY_SEV:
    comp, pr = setting_comp_pr(setting)
    a = results[setting]["acc"]
    k = a.bin_deepest()
    if k is None:
        continue
    dt = a.rel_tilt(k)[TILT_BAND]
    slope = np.polyfit(logf, dt, 1)[0]      # dB per decade
    rows.append({
        "Setting": setting,
        "Mode": "Compress" if comp == 0 else "Limit",
        "PeakRed": pr,
        "depth dB": round(a.mean_depth(k), 2),
        "frames@depth": int(a.nframes_k[k]),
        "HF-LF gain dB": round(a.hf_lf_gain(k), 3),
        "tilt slope dB/dec": round(slope, 3),
        "overall HF-LF dB": round(float(np.mean(a.gain_overall()[HF_BAND])
                                         - np.mean(a.gain_overall()[LF_BAND])), 3),
    })
summary_df = pd.DataFrame(rows)
display(summary_df.reset_index(drop=True))